# F-08 Tier-1 Medium-Scale Full-Threat Run — Default Allocator

This notebook is the copied/adapted successor to the proven Paper8 allocator notebook. It retains the one-allocator `ExperimentConfiguration` + `AllocatorRunner` workflow and executes the complete five-threat spectrum. Each completed notebook references 45 immutable scientific bundles written outside the source repository.


## Environment Setup & Library Installation


In [ ]:
import os
import sys
from pathlib import Path

try:
    import google.colab
    from google.colab import drive
    drive.mount('/content/drive')
    project_dir = Path('/content/drive/MyDrive/GA-Work/hybrid_variable_framework/Dynamic_Routing_Eval_Framework')
    os.chdir(project_dir)
except ImportError:
    project_dir = Path.cwd()

sys.path.insert(0, str(Path.cwd()))

from daqr.config.experiment_config import ExperimentConfiguration
from daqr.campaigns.medium_scientific import (
    AllocatorRunner,
    TIER1_MODELS,
    TIER1_SCENARIOS,
    build_tier1_default_config,
)

print(f'Framework root: {Path.cwd()}')


## Frozen Tier-1 Run Configuration


In [ ]:
SOURCE_NOTEBOOK_COMMIT = '96b327de7e3571ad3cbf05bbeee02cfb922ca2c3'
SOURCE_NOTEBOOK_BLOB = '599bb49b58a41fc98ff7d6f10c4077c941507269'
SOURCE_NOTEBOOK_SHA256 = '714eefbd1eafc3c66347fc6b0460e6aaf706ad25b1ea411bcc89ac317a4711c9'

allocator_type = 'Default'
models = list(TIER1_MODELS)
test_scenarios = dict(TIER1_SCENARIOS)
config = build_tier1_default_config()
assert isinstance(config, ExperimentConfiguration)
assert list(test_scenarios) == ['stochastic', 'markov', 'adaptive', 'onlineadaptive', 'none']
assert config.runs * len(models) * len(test_scenarios) == 45

OUTPUT_ROOT = Path(os.environ.get(
    'QUANTUM_MEDIUM_OUTPUT_ROOT',
    '/content/drive/MyDrive/GA-Work/quantum_experiment_evidence/medium-tier1/default-fixed',
)).expanduser().resolve()
MAX_WORKERS = int(os.environ.get('QUANTUM_MEDIUM_MAX_WORKERS', '1'))

print('Allocator:', allocator_type)
print('Policies:', models)
print('Threat order:', list(test_scenarios))
print('Blocks:', config.runs)
print('Frames per cell:', config.execution.horizon)
print('Required cells:', config.runs * len(models) * len(test_scenarios))
print('External evidence root:', OUTPUT_ROOT)
print('Process workers:', MAX_WORKERS)


## Run One Allocator Across the Complete Threat Spectrum


In [ ]:
alloc_runner = AllocatorRunner(
    allocator_type=allocator_type,
    output_root=OUTPUT_ROOT,
    max_workers=MAX_WORKERS,
    config=config,
)
receipt = alloc_runner.run()

assert receipt['required_cells'] == 45
assert receipt['completed_cells'] == 45
assert receipt['scenario_order'] == ['stochastic', 'markov', 'adaptive', 'onlineadaptive', 'none']

print('COMPLETE:', receipt['completed_cells'], '/', receipt['required_cells'])
print('Notebook receipt:', receipt['receipt_path'])


## Evidence Boundary

- The notebook is complete evidence only when all 45 immutable bundles validate.
- Valid poor or zero results are retained; no performance-triggered reruns are permitted.
- The fixed-allocator notebook alone does not establish allocator sensitivity.
- DynamicUCB and ThompsonSampling remain on hold; Random remains conditional.
